In [1]:
import tensorflow as tf

model = tf.keras.models.load_model('sports.keras')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 576)            │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        57,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,112,222 (4.24 MB)

 Trainable params: 57,700 (225.39 KB)

 Non-trainable params: 939,120 (3.58 MB)

 Optimizer params: 115,402 (450.79 KB)

In [2]:
import kagglehub

path = kagglehub.dataset_download('gpiosenka/sports-classification')

100%|██████████| 424M/424M [00:17<00:00, 25.9MB/s]

Extracting files...


In [5]:
import os

train_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(path, 'train'),
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    batch_size=100,
    image_size=(224, 224),
    shuffle=True,
)

Found 13492 files belonging to 100 classes.


In [8]:
import numpy as np


x_train, y_train = next(iter(train_data))

def representative_data_gen():
    # Use about 100 samples from your training/val set
    # Assuming 'x_train' is your numpy array of images
    for i in range(100):
        data = np.expand_dims(x_train[i], axis=0).astype(np.float32)
        yield [data]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# This forces the model to be FULL integer (Input, Weights, and Output)
# This is what allows the ESP32-S3 to use its SIMD instructions for speed
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8   # Optional: keep as float32 if you don't want to manual-scale
converter.inference_output_type = tf.int8 # Optional

tflite_model = converter.convert()

with open("sports_model.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpk11iexiv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 100), dtype=tf.float32, name=None)
Captures:
  139331250822544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250824080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250824848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250823120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250822928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250825040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250826384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250827344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250826960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139331250826192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13933125082

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [9]:
!xxd -i sports_model.tflite > sports.h